In [2]:
def norm(val):
    """Normalise practice/facility for matching.
    '4013 - Suffolk, VA (Harborview)' → '4013 suffolk va'
    '4013- Suffolk, VA'               → '4013 suffolk va'
    '4035 - Los Lunas, NM - closed'   → '4035 los lunas nm'
    """
    s = (val or "").strip().lower()
    s = re.sub(r"\(.*?\)", "", s)                # strip parenthetical like (Harborview)
    s = re.sub(r"\s*-\s*(closed|closing)\s*$", "", s)  # strip trailing - Closed
    s = re.sub(r"[,.\-–]", " ", s)               # replace punctuation with spaces (keeps store number as token)
    return re.sub(r"\s+", " ", s).strip()

def norm_date(val):
    s = (val or "").strip()
    if not s:
        return ""
    if len(s) >= 10 and s[4] == "-":
        return s[:10]
    m = re.match(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
    if m:
        return f"{m.group(3)}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    m = re.match(r"(\d{1,2})/(\d{1,2})/(\d{2})$", s)
    if m:
        yr = 2000 + int(m.group(3))
        return f"{yr}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    return s[:10]

_ADDR_STRIP = re.compile(r"\b(st|street|ave|avenue|blvd|boulevard|dr|drive|rd|road|ln|lane|ct|court|cir|circle|way|pl|place|ste|suite|apt|unit|#)\b", re.I)
_ADDR_NOISE = re.compile(r"[.,#\-/]")

def norm_addr(val):
    """Normalise address for fuzzy comparison: lowercase, strip common suffixes/noise, collapse whitespace."""
    s = (val or "").strip().lower()
    s = _ADDR_NOISE.sub(" ", s)
    s = _ADDR_STRIP.sub("", s)
    return re.sub(r"\s+", " ", s).strip()

def addr_token_overlap(a, b):
    """Jaccard similarity on address tokens (0..1)."""
    ta = set(a.split())
    tb = set(b.split())
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

ADDR_THRESHOLD = 0.5

# ---- Build SF lookup indices ----
sf_by_practice = defaultdict(set)
sf_by_date = defaultdict(set)
sf_by_both = defaultdict(set)
sf_addr_index = []  # list of (norm_addr, sf_id) for fuzzy matching

for r in sf_jobs:
    p = norm(r.get("Job_Client_Job_Id__c"))
    d = norm_date(r.get("Job_Open_Date__c"))
    a = norm_addr(r.get("Job_Worksite_1_Address__c"))
    sid = r["Id"]
    if p:
        sf_by_practice[p].add(sid)
    if d:
        sf_by_date[d].add(sid)
    if p and d:
        sf_by_both[(p, d)].add(sid)
    if a:
        sf_addr_index.append((a, sid))

def find_addr_matches(kim_addr):
    """Return set of SF Ids whose address overlaps above threshold."""
    if not kim_addr:
        return set()
    return {sid for sa, sid in sf_addr_index if addr_token_overlap(kim_addr, sa) >= ADDR_THRESHOLD}

# ---- Strategies ----
strategy_defs = [
    ("practice only",          True,  False, False),
    ("date only",              False, True,  False),
    ("address only",           False, False, True),
    ("practice + date",        True,  True,  False),
    ("practice + address",     True,  False, True),
    ("date + address",         False, True,  True),
    ("practice + date + addr", True,  True,  True),
]

rows = []
for label, use_p, use_d, use_a in strategy_defs:
    c1, cN, c0 = 0, 0, 0
    for kr in kim_rows:
        p = norm(kr.get("practice_value"))
        d = norm_date(kr.get("posted_date"))
        a = norm_addr(kr.get("address_line"))

        needed = []
        if use_p: needed.append(p)
        if use_d: needed.append(d)
        if use_a: needed.append(a)
        if not all(needed):
            c0 += 1
            continue

        hit_sets = []
        if use_p: hit_sets.append(sf_by_practice.get(p, set()))
        if use_d: hit_sets.append(sf_by_date.get(d, set()))
        if use_a: hit_sets.append(find_addr_matches(a))

        hits = hit_sets[0]
        for hs in hit_sets[1:]:
            hits = hits & hs

        if len(hits) == 1:   c1 += 1
        elif len(hits) > 1:  cN += 1
        else:                c0 += 1

    rows.append({"strategy": label, "1:1": c1, "1:N": cN, "unmatched": c0, "total": len(kim_rows)})

print(f"{'Strategy':<28} {'1:1':>6} {'1:N':>6} {'None':>6} {'Total':>6}")
print("-" * 56)
for r in rows:
    print(f"{r['strategy']:<28} {r['1:1']:>6} {r['1:N']:>6} {r['unmatched']:>6} {r['total']:>6}")

Strategy                        1:1    1:N   None  Total
--------------------------------------------------------
practice only                   103      0      7    110
date only                        22     53     35    110
address only                     86      9     15    110
practice + date                   3      0    107    110
practice + address               88      0     22    110
date + address                    3      0    107    110
practice + date + addr            3      0    107    110


In [3]:
# Sample non-null values being compared (raw → normalised)
print(f"{'Source':<12} {'Field':<30} {'Raw → Normalised'}")
print("-" * 100)

kim_practices = [(r.get("practice_value") or "") for r in kim_rows if (r.get("practice_value") or "").strip()][:8]
kim_dates = [(r.get("posted_date") or "") for r in kim_rows if (r.get("posted_date") or "").strip()][:5]
kim_addrs = [(r.get("address_line") or "") for r in kim_rows if (r.get("address_line") or "").strip()][:5]
sf_practices = [(r.get("Job_Client_Job_Id__c") or "") for r in sf_jobs if (r.get("Job_Client_Job_Id__c") or "").strip()][:8]
sf_dates = [(r.get("Job_Open_Date__c") or "") for r in sf_jobs if (r.get("Job_Open_Date__c") or "").strip()][:5]
sf_addrs = [(r.get("Job_Worksite_1_Address__c") or "") for r in sf_jobs if (r.get("Job_Worksite_1_Address__c") or "").strip()][:5]

for label, field, vals, fn in [
    ("Kimedics", "practice_value", kim_practices, norm),
    ("Salesforce", "Job_Client_Job_Id__c", sf_practices, norm),
    ("Kimedics", "posted_date", kim_dates, norm_date),
    ("Salesforce", "Job_Open_Date__c", sf_dates, norm_date),
    ("Kimedics", "address_line", kim_addrs, norm_addr),
    ("Salesforce", "Job_Worksite_1_Address__c", sf_addrs, norm_addr),
]:
    pairs = [f"{v} → {fn(v)}" for v in vals]
    print(f"{label:<12} {field:<30} {' | '.join(pairs)}")

Source       Field                          Raw → Normalised
----------------------------------------------------------------------------------------------------
Kimedics     practice_value                 2350 - Lawton, OK → 2350 lawton ok | 4292 - Tomball, TX → 4292 tomball tx | 1307 - Battle Creek, MI → 1307 battle creek mi | 4180 - Shallotte, NC → 4180 shallotte nc | 1062 - Springfield, IL (West) → 1062 springfield il | 1314 - Springfield, IL → 1314 springfield il | 4339 - Clearwater, FL → 4339 clearwater fl | 4255 - Burlington, NJ → 4255 burlington nj
Salesforce   Job_Client_Job_Id__c           2022-58405 → 2022 58405 | 2022-58812 → 2022 58812 | 2022-57867 → 2022 57867 | 2022-57455 → 2022 57455 | 3163 - Santa Fe, NM → 3163 santa fe nm | 4043- Hobbs, NM → 4043 hobbs nm | 1311- Troy, OH → 1311 troy oh | 2402- Amarillo, TX → 2402 amarillo tx
Kimedics     posted_date                    03/24/26 → 2026-03-24 | 03/30/26 → 2026-03-30 | 03/12/26 → 2026-03-12 | 03/16/26 → 2026-03-16 | 02/2

In [4]:
# Anchor: practice_value. For unmapped rows, show potential matches from other signals.
sf_by_id = {r["Id"]: r for r in sf_jobs}

def sf_job_nums(ids):
    """Format SF Ids as Job_Number_DJC__c list."""
    nums = [(sf_by_id[s].get("Job_Number_DJC__c") or s[:10]) for s in sorted(ids) if s in sf_by_id]
    return ", ".join(nums[:5]) + ("..." if len(nums) > 5 else "")

def sf_addrs(ids):
    """Return SF addresses for matched Ids."""
    addrs = [(sf_by_id[s].get("Job_Worksite_1_Address__c") or "") for s in sorted(ids) if s in sf_by_id]
    unique = list(dict.fromkeys(a for a in addrs if a.strip()))
    return " | ".join(unique[:3]) + ("..." if len(unique) > 3 else "")

unmapped = []
for kr in kim_rows:
    p = norm(kr.get("practice_value"))
    d = norm_date(kr.get("posted_date"))
    a = norm_addr(kr.get("address_line"))

    practice_hits = sf_by_practice.get(p, set()) if p else set()
    if len(practice_hits) == 1:
        continue  # mapped 1:1 by practice — skip

    # Potential matches via other signals
    addr_hits = find_addr_matches(a) if a else set()
    date_hits = sf_by_date.get(d, set()) if d else set()
    date_addr_hits = (date_hits & addr_hits) if (d and a) else set()

    unmapped.append({
        "kr": kr,
        "reason": "missing_practice" if not p else ("no_match" if not practice_hits else f"{len(practice_hits)}_matches"),
        "addr_hits": addr_hits,
        "date_addr_hits": date_addr_hits,
    })

print(f"Practice anchor  →  {len(unmapped)} unmapped out of {len(kim_rows)}\n")
import pandas as pd

rows_for_df = []
for row in unmapped:
    kr = row["kr"]
    rows_for_df.append({
        "job_id": kr.get("job_id") or "",
        "status": kr.get("status") or "",
        "practice_value": kr.get("practice_value") or "",
        "posted_date": kr.get("posted_date") or "",
        "by_addr": sf_job_nums(row["addr_hits"]),
        "addr_1to1": "✅" if len(row["addr_hits"]) == 1 else "❌",
        "SF_address": sf_addrs(row["addr_hits"]),
        "by_date+addr": sf_job_nums(row["date_addr_hits"]),
        "address_line": kr.get("address_line") or "",
        "view_job_link": kr.get("view_job_link") or "",
    })

df = pd.DataFrame(rows_for_df)

def make_link(url):
    if url:
        return f'<a href="{url}" target="_blank">link</a>'
    return ""

df.style.format({"view_job_link": make_link}).set_properties(**{"text-align": "left"})


Practice anchor  →  7 unmapped out of 110



,job_id,status,practice_value,posted_date,by_addr,addr_1to1,SF_address,by_date+addr,address_line,view_job_link
0,19553,Closed,"4373 - Tifton, GA",03/27/26,,❌,,,"1303 US-82, Tifton, GA",link
1,19531,Closed,"4439 - Midlothian, TX",03/23/26,JN-032026-4813,✅,"110 Eric Street, Midlothian, TX 76065",,"110 Eric Street, Midlothian, TX",link
2,19437,Closed,"4247 - Houston, TX (NW Crossing)",02/26/26,JN-042023-2127,✅,"4530 Dacoma Street, Houston, TX 77092",,"4530 DACOMA ST UNIT 500, HOUSTON TX",link
3,19453,Closed,"3185 - St. Joseph, MO",03/03/26,JN-022022-1335,✅,"5101 North Belt Hwy, St. Joseph, MO",,"5101 N BELT HWY, SAINT JOSEPH MO, St. Joseph, MO",link
4,19455,Closed,"4399 - Carson City, NV",03/03/26,JN-062025-4362,✅,"3815 South Carson Street, Carson City, NV 89701",,3815 S. Carson St. Carson City NV,link
5,19488,Closed,"3185 - St. Joseph, MO",03/10/26,JN-022022-1335,✅,"5101 North Belt Hwy, St. Joseph, MO",,"5101 N BELT HWY, SAINT JOSEPH MO, St. Joseph, MO",link
6,19523,Closed,"4140 - Suffolk, VA",03/19/26,JN-092023-2369,✅,"1930 North Main Street, Suffolk, VA 23434",,"1930 N MAIN ST, Suffolk, VA",link
